# 03 — Seq2Seq Model Comparison

Interactively fits and inspects individual seq2seq models on one farm.
Useful for sanity-checking a single model before running the full benchmark.

For the full benchmark run:
```bash
python scripts/run_seq2seq.py --config experiments/configs/seq2seq.yaml
```

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from windbench.data.loader import load_all_farms
from windbench.data.seq2seq import (
    build_seq2seq_arrays,
    seq2seq_temporal_split,
    build_production_hour_arrays,
)
from windbench.evaluation.metrics import evaluate

sns.set_theme(style='whitegrid')
%matplotlib inline

RAW_DIR   = '../data/raw'
TARGET_COL = 'energy_total'

In [ ]:
farms = load_all_farms(RAW_DIR, target_col=TARGET_COL)
farm_name = list(farms)[0]
df = farms[farm_name]

print(f'Farm: {farm_name}   shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

## Build seq2seq arrays

One NWP run = one row.  Each run has T=72 lead-time steps and F weather features.

In [ ]:
weather_cols = [c for c in ['2d','2t','10u','10v','100u','100v','msl','tp'] if c in df.columns]

X, y, run_times = build_seq2seq_arrays(
    df,
    weather_cols=weather_cols,
    target_col=TARGET_COL,
    min_lead_coverage=0.80,
    fill_method='interpolate',
)
print(f'X: {X.shape}   y: {y.shape}   runs: {len(run_times)}')
print(f'Run date range: {run_times[0].date()} → {run_times[-1].date()}')

In [ ]:
(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    t_train, t_val, t_test,
) = seq2seq_temporal_split(X, y, run_times, val_frac=0.10, test_frac=0.20)

print(f'Train: {len(X_train)} runs   Val: {len(X_val)}   Test: {len(X_test)}')

## Production-hour view (tree models)

For each production hour, up to 6 NWP forecasts from different runs are gathered,
ordered freshest → oldest.  The feature density is much higher than the flat-run approach.

In [ ]:
X_ph, y_ph, ph_times, ph_leads = build_production_hour_arrays(
    X_train, y_train, t_train, n_lags=6
)
print(f'Production-hour arrays:  X_ph={X_ph.shape}  y_ph={y_ph.shape}')
print(f'Available lead times for first production hour: {ph_leads[0][~np.isnan(ph_leads[0])]}')
print(f'NaN fraction in X_ph: {np.isnan(X_ph).mean():.1%}')

## Fit a single model manually

In [ ]:
from windbench.models.seq2seq_trees import Seq2SeqRandomForestModel

model = Seq2SeqRandomForestModel(n_estimators=100, random_state=42)
model.fit(X_train, y_train, run_times=t_train)

EVAL_HORIZONS = [1, 6, 24]
lead_times = list(range(1, 73))

h_preds = model.predict_at_horizons(X_test, EVAL_HORIZONS, lead_times, run_times=t_test)

for h in EVAL_HORIZONS:
    h_idx  = lead_times.index(h)
    y_true = y_test[:, h_idx]
    y_pred = h_preds[h]
    base   = np.full(len(y_true), float(np.nanmean(y_train[:, h_idx])))
    m      = evaluate(y_true, y_pred, base)
    print(f'h={h:2d}  RMSE={m["rmse"]:8.1f}  skill={m["skill_score"]:+.3f}')

## Prediction vs truth — scatter per horizon

In [ ]:
fig, axes = plt.subplots(1, len(EVAL_HORIZONS), figsize=(5 * len(EVAL_HORIZONS), 4))

for ax, h in zip(axes, EVAL_HORIZONS):
    h_idx  = lead_times.index(h)
    y_true = y_test[:, h_idx]
    y_pred = h_preds[h]

    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    ax.scatter(y_true[mask], y_pred[mask], alpha=0.3, s=8, color='steelblue')
    lim = [min(y_true[mask].min(), y_pred[mask].min()),
           max(y_true[mask].max(), y_pred[mask].max())]
    ax.plot(lim, lim, 'k--', linewidth=0.8)
    ax.set_title(f'h={h}h')
    ax.set_xlabel('Actual (kWh/h)')
    ax.set_ylabel('Predicted (kWh/h)')

fig.suptitle(f'Random Forest — {farm_name}', fontsize=12)
fig.tight_layout()
plt.show()

## Compare multiple models at one horizon

In [ ]:
from windbench.models.seq2seq_trees import Seq2SeqXGBoostModel
from windbench.models.deep.seq2seq_lstm import Seq2SeqLSTMModel

COMPARE_HORIZON = 24
h_idx = lead_times.index(COMPARE_HORIZON)

models_to_compare = {
    'Random Forest': Seq2SeqRandomForestModel(n_estimators=100),
    'XGBoost':       Seq2SeqXGBoostModel(n_estimators=100),
    'LSTM':          Seq2SeqLSTMModel(hidden_dim=64, epochs=20),
}

results_cmp = []
for name, m in models_to_compare.items():
    print(f'Fitting {name}...')
    m.fit(X_train, y_train, X_val=X_val, y_val=y_val, run_times=t_train)
    preds = m.predict_at_horizons(X_test, [COMPARE_HORIZON], lead_times, run_times=t_test)
    y_true = y_test[:, h_idx]
    y_pred = preds[COMPARE_HORIZON]
    base   = np.full(len(y_true), float(np.nanmean(y_train[:, h_idx])))
    metrics = evaluate(y_true, y_pred, base)
    results_cmp.append({'model': name, **metrics})

cmp_df = pd.DataFrame(results_cmp).set_index('model')[['rmse', 'mae', 'skill_score']].round(3)
print(f'\n── h={COMPARE_HORIZON}h comparison ──')
cmp_df

## Time-series plot — first 2 weeks of test set

In [ ]:
# Use the models fitted in the cell above
n_show = min(28, len(X_test))   # 28 runs = 2 weeks (12h runs)

fig, ax = plt.subplots(figsize=(14, 4))

# Ground truth — one point per test run at h=COMPARE_HORIZON
y_true_show = y_test[:n_show, h_idx]
ax.plot(range(n_show), y_true_show, 'k-', linewidth=1.5, label='Ground truth')

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
for (name, m), color in zip(models_to_compare.items(), colors):
    preds = m.predict_at_horizons(
        X_test[:n_show], [COMPARE_HORIZON], lead_times,
        run_times=t_test[:n_show]
    )
    ax.plot(range(n_show), preds[COMPARE_HORIZON], '--', color=color, linewidth=1.2, label=name)

ax.set_title(f'{farm_name} — h={COMPARE_HORIZON}h — first {n_show} test runs')
ax.set_xlabel('Test run index')
ax.set_ylabel('Energy (kWh/h)')
ax.legend()
fig.tight_layout()
plt.show()